In [ ]:
# ==============================================================
# 必要なライブラリの追加インストール
# ==============================================================
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart transformers torch accelerate pypdf chardet uvicorn
print("ライブラリインストール完了。")
import os
import io
import zipfile
import chardet
import uvicorn
from pypdf import PdfReader
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import nest_asyncio
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
# ==============================================================
# 設定・モデル読み込み
# ==============================================================
NGROK_AUTH_TOKEN = "35dNjobYaFZhkE8WBJ8nLhKfS7D_TvQqfmyTrpj2Twphz2Hz"
# AIモデルの設定
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"AIモデル ({MODEL_NAME}) をロード中...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
print("AIモデル ロード完了")
app = FastAPI()
PROMPT_TEMPLATE = """あなたは日本のIT企業の技術責任者です。
必ず「日本語のみ」で、厳格かつ具体的にコードを査定してください。

【査定ルール】
1. 最初に必ず [[SCORE]] ブロックを書いてください。
2. その後、■分析レポート を書いてください。
3. コードが完璧な場合は、無理に欠点を探さず「非常に優れた設計です」と評価してください。

[[SCORE]]
正確性: (数値)
可読性: (数値)
パフォーマンス: (数値)
スタイル: (数値)
[[END]]

■分析レポート
（ファイル名、問題点、改善策を日本語で記述）

【対象コード】
{code}
"""
# ファイルの内容をテキストとして抽出するヘルパー関数
def extract_text_from_file(file_content: bytes, filename: str) -> str:
    ext = filename.split('.')[-1].lower()
    
    # PDFの場合
    if ext == 'pdf':
        try:
            reader = PdfReader(io.BytesIO(file_content))
            text = ""
            for page in reader.pages:
                text += page.extract_text() + "\n"
            return text
        except Exception as e:
            return f"PDF解析エラー: {str(e)}"
    
    # ZIPの場合
    elif ext == 'zip':
        try:
            text_all = ""
            with zipfile.ZipFile(io.BytesIO(file_content)) as z:
                for info in z.infolist():
                    if not info.is_dir() and info.filename.endswith(('.py', '.html', '.css', '.js', '.txt', '.md')):
                        with z.open(info.filename) as f:
                            raw = f.read()
                            encoding = chardet.detect(raw)['encoding'] or 'utf-8'
                            try:
                                decoded = raw.decode(encoding, errors='ignore')
                                text_all += f"--- File: {info.filename} ---\n{decoded}\n\n"
                            except:
                                pass
            if not text_all:
                return "ZIP内に読み取れるテキストファイル（.py, .html, .txt等）が見つかりませんでした。"
            return text_all
        except Exception as e:
            return f"ZIP解析エラー: {str(e)}"
    # それ以外（テキストファイルとみなす）
    else:
        encoding = chardet.detect(file_content)['encoding'] or 'utf-8'
        try:
            return file_content.decode(encoding, errors='ignore')
        except:
            return file_content.decode('utf-8', errors='ignore')
@app.get("/")
def home():
    return {"status": "ok", "message": "AI Server is ready."}
@app.post("/grade_file/")
async def grade_file(upload: UploadFile = File(...)):
    try:
        content = await upload.read()
        print(f"ファイル受信: {upload.filename}")
        code_text = extract_text_from_file(content, upload.filename)
        
        if not code_text or len(code_text.strip()) < 10:
            return JSONResponse({
                "filename": upload.filename, 
                "result": "エラー: ファイルからテキストを読み取れませんでした。",
                "preview": ""
            })
        # コードが長すぎる場合は先頭3000文字にカット
        input_text = PROMPT_TEMPLATE.format(code=code_text[:3000])
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": input_text}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=512,
                temperature=0.0,
                do_sample=False
            )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        print("採点完了")
        
        return JSONResponse(content={
            "filename": upload.filename, 
            "result": response_text,
            "preview": code_text[:2000] 
        })
    except Exception as e:
        print(f"Error: {e}")
        return JSONResponse(content={"error": str(e)}, status=500)
if __name__ == "__main__":
    if NGROK_AUTH_TOKEN == "ここにトークンを貼り付け":
         print("エラー: NGROK_AUTH_TOKEN が設定されていません")
    else:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        tunnel = ngrok.connect(8000)
        public_url = tunnel.public_url
        print(f"\n========================================================")
        print(f"AIサーバー起動成功！")
        print(f"以下のURLをDjangoの views.py にコピーしてください:")
        print(f"   {public_url}/grade_file/")
        print(f"========================================================\n")
        nest_asyncio.apply()
        config = uvicorn.Config(app, host="0.0.0.0", port=8000)
        server = uvicorn.Server(config)
        await server.serve()
